# Session 4 — Composite Systems & Channels

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

Two systems, not one. We combine qubits with the **tensor product**, recover a part with
the **partial trace** (the quantum *marginal*), meet **entanglement** (a pure whole with
mixed parts), and describe noisy evolution as a **quantum channel**. This closes Movement 1.

## 0. What you'll be able to do

- Combine systems with the **tensor product**.
- Take a **partial trace** and see it *is* the quantum marginal.
- Recognise **entanglement**: a Bell state is pure, yet each qubit alone is maximally mixed.
- Describe noise as a **quantum channel** (CPTP / Kraus) and watch it mix a state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.states import KET_0, KET_PLUS
from qot_course.quantum.density import (
    density_matrix, maximally_mixed, purity, von_neumann_entropy,
)
from qot_course.quantum.composite import (
    tensor, partial_trace, bell_state, entanglement_entropy,
    apply_channel, depolarizing_channel, is_cptp,
)

np.random.seed(42)
viz.use_course_style()

## 1. Two systems: the tensor product

To describe two qubits together we combine their spaces with the **tensor product**
$\mathcal{H}_A \otimes \mathcal{H}_B$ (dimension $2 \times 2 = 4$). States and operators
combine by the Kronecker product. Let's build $|0\rangle \otimes |+\rangle$.

In [ ]:
state_AB = tensor(KET_0, KET_PLUS)
rho_AB = density_matrix(state_AB)
print("composite state dimension:", state_AB.shape[0])
viz.plot_density_matrix(rho_AB, title="rho for |0> ⊗ |+>  (a product state)")
plt.show()

**Read the figure.** A $4\times4$ density matrix indexed by the basis
$|00\rangle, |01\rangle, |10\rangle, |11\rangle$. The mass sits only in the top-left block
because qubit A is $|0\rangle$; within it, the $|+\rangle$ coherences of qubit B appear.

## 2. The partial trace = the quantum marginal

Remember the classical **marginal** (sum a joint distribution over one variable)? Its
quantum version is the **partial trace**: $\rho_A = \mathrm{tr}_B\,\rho_{AB}$ gives the
state of qubit A on its own. Let's recover both factors.

In [ ]:
reduced_A = partial_trace(rho_AB, keep=[0], dims=[2, 2])
reduced_B = partial_trace(rho_AB, keep=[1], dims=[2, 2])
print("reduced A == |0>? ", np.allclose(reduced_A, density_matrix(KET_0)))
print("reduced B == |+>? ", np.allclose(reduced_B, density_matrix(KET_PLUS)))

**Read the output.** Tracing out B recovers A *exactly* (and vice versa) — for a product
state, the parts are perfectly well-defined, just like marginalising independent variables.
That is about to change.

## 3. Entanglement

The **Bell state** $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$ cannot be written as a
product. Watch what the partial trace does to it.

In [ ]:
rho_bell = density_matrix(bell_state())
reduced_bell_A = partial_trace(rho_bell, keep=[0], dims=[2, 2])

print("global purity (Bell):", round(purity(rho_bell), 3), " -> pure")
print("reduced A == I/2? ", np.allclose(reduced_bell_A, maximally_mixed(2)))
print("entanglement entropy:", round(entanglement_entropy(rho_bell, dims=[2, 2]), 3), "bit")

viz.plot_density_matrix(rho_bell, title="Bell state rho (global) — pure")
viz.plot_density_matrix(reduced_bell_A, title="Reduced state of qubit A — I/2 (maximally mixed)")
plt.show()

**The punchline — read both figures.** The global Bell state is **pure** (purity 1, the
off-diagonal corners show the $|00\rangle\!-\!|11\rangle$ coherence). Yet each qubit *alone*
is **maximally mixed** ($I/2$, 1 bit of entropy). The whole is more defined than its parts —
something **no classical joint distribution can do**. That is entanglement, and it is the
resource that makes quantum information genuinely different.

## 4. Quantum channels: noisy evolution

The quantum version of a **Markov kernel** is a **quantum channel** — a completely-positive,
trace-preserving (CPTP) map $\rho \mapsto \sum_k K_k \rho K_k^\dagger$ with
$\sum_k K_k^\dagger K_k = I$. The **depolarizing channel** mixes a state toward $I/2$. Let's
push $|+\rangle$ through increasing noise.

In [ ]:
print("depolarizing channel is CPTP:", is_cptp(depolarizing_channel(0.3)))

ps = np.linspace(0.0, 1.0, 11)
rho_plus = density_matrix(KET_PLUS)
purities = [purity(apply_channel(rho_plus, depolarizing_channel(p))) for p in ps]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ps, purities, "-o", color=viz.FLOW_COLOR, markersize=8)
ax.axhline(0.5, color=viz.TARGET_COLOR, linestyle="--", label="I/2 (fully mixed)")
ax.set_xlabel("depolarizing parameter p")
ax.set_ylabel("purity  tr(rho$^2$)")
ax.set_title("A quantum channel mixes a state toward I/2", pad=12)
ax.legend()
plt.show()

**Read the figure.** As the noise $p$ grows, purity falls from 1 (pure $|+\rangle$) toward
0.5 (the maximally mixed $I/2$). This is exactly the noise that turned S3's tomographed
$|+\rangle$ into a *mixed* state — now we can name it: a quantum channel acting on the qubit.

## 5. Dictionary update

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | Born rule $|\langle x|\psi\rangle|^2$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| Markov kernel | **quantum channel (CPTP map)** |
| independent variables | **product state (no entanglement)** |

## 6. Your turn

1. **Other Bell states.** Build $\tfrac{1}{\sqrt2}(|01\rangle + |10\rangle)$ and check its
   reduced states and entanglement entropy. Same as the $|00\rangle+|11\rangle$ Bell state?
2. **Partial vs full noise.** Apply `depolarizing_channel(p)` to one qubit of the Bell state
   (tensor with identity) and watch the entanglement entropy drop as $p$ grows.
3. **Bit-flip channel.** Write the Kraus operators of the bit-flip channel
   $\{\sqrt{1-p}\,I, \sqrt{p}\,X\}$, check `is_cptp`, and compare its effect to depolarizing.

## Conclusions & references

- **Tensor product** combines systems; **partial trace** is the quantum marginal.
- **Entanglement**: a pure whole with maximally mixed parts — impossible classically.
- **Quantum channels** (CPTP/Kraus) are the quantum Markov kernels; noise mixes states.
- **Movement 1 complete.** Next (**S5 — Classical information theory**) we start the spine:
  entropy, KL divergence, mutual information, transfer entropy.

**References:** Nielsen & Chuang ch. 2 & 8; M. Wilde, *Quantum Information Theory*, ch. 4. Previous: `notebooks/s03_density.ipynb`.